In [2]:
import openmm.unit as unit
import torch

In [77]:
# Ballpark estimating what gamma we were using. 
# Assuming temp = 300, delta t = 0.002 ps = 2e-15 s, carbon atom (mass = 1.9944733 × 10^-26 kg)
# 1e-5 A^2 = 1e−25 m^2

epsilon = 1e-6

kT = unit.constants.BOLTZMANN_CONSTANT_kB * 300 * unit.kelvin # Joules.
kT_converted = kT._value * unit.kilogram * unit.meter ** 2 / (unit.second ** 2)

gamma = kT_converted / (1.9944733e-26 * unit.kilogram * epsilon * 1e-20 * unit.meter ** 2) * 2e-15 * unit.second
gamma.in_units_of(unit.picosecond ** -1)

41534.24365219631 /ps

In [50]:
batch = 10
atom = 100
dim = 3
temperature = 300
friction = 10

time_units = "picosecond"
temperature_units = "kelvin"
energy_units = "kilojoules_per_mole"
energy_units = "kilojoules"
length_units = "angstrom"

temperature_omm = temperature * getattr(unit, temperature_units)
friction_omm = friction / getattr(unit, time_units)

In [9]:
unit.AVOGADRO_CONSTANT_NA * unit.mole

6.02214076e+23

In [31]:
x = torch.rand(batch, atom, dim)
masses = torch.rand(batch, atom, dim)
masses_np = masses.detach().cpu().numpy()

'''Masses are computed in boltz1.py and are initially in amu 
(g/mol), which is equivalent to 
(kJ * s^2) / (m^2 * mol) (unit_mass_omm). 

During integration, we will convert masses to  
unit_energy_omm * unit_time_omm ** 2 / unit_length_omm ** 2. To do so, 
in the case that unit_energy is not per-mole, we must perform an 
initial conversion of the mass units to keep calculations 
consistent during integration. In this case, we scale masses so
that they are in units of g, which is equivalent to (kJ * s^2) / m^2.
'''

if getattr(unit, energy_units).is_compatible(unit.kilojoule_per_mole):
    kT_omm = (unit.AVOGADRO_CONSTANT_NA 
            * unit.BOLTZMANN_CONSTANT_kB 
            * temperature_omm) # J / mol.
    mass_unit = unit.kilojoule * unit.second ** 2 / (unit.meter ** 2 * unit.mole)
    print(mass_unit)
else:
    kT_omm = unit.BOLTZMANN_CONSTANT_kB * temperature_omm # J.
    masses_np /= unit.AVOGADRO_CONSTANT_NA * unit.mole # Scaling masses from g/mol to g.
    mass_unit = unit.kilojoule * unit.second ** 2 / (unit.meter ** 2)
    print(mass_unit)

masses_omm = masses_np * mass_unit
target_unit = (getattr(unit, energy_units) * getattr(unit, time_units)**2 
                    / getattr(unit, length_units)**2)
masses_omm_converted = masses_omm.in_units_of(target_unit)

per_atom_gm = masses_omm_converted * friction_omm # unit_energy_omm * unit_time_omm ** 2 / unit_length_omm ** 2
diffusion_constants = kT_omm / per_atom_gm # length^2 / time
print(diffusion_constants.unit)
print(diffusion_constants.unit == getattr(unit, length_units) ** 2 / getattr(unit, time_units))

second**2*kilojoule/(meter**2)
angstrom**2/picosecond
True


In [37]:
if getattr(unit, energy_units).is_compatible(unit.kilojoule_per_mole):
    print("Energy units are compatible with kJ/mol.")
    beta = (
        1 / (
            temperature * getattr(unit, temperature_units) 
            * unit.AVOGADRO_CONSTANT_NA 
            * unit.BOLTZMANN_CONSTANT_kB
        )
    ).value_in_unit(getattr(unit, energy_units)) # 1 / (T * mol^-1 * J/T) = mol / J (inverse self.energy_units).
    print(beta)
else:
    print("Energy units are not compatible with kJ/mol.")
    beta = (
        1 / (
            temperature * getattr(unit, temperature_units) 
            * unit.BOLTZMANN_CONSTANT_kB
        ).value_in_unit(getattr(unit, energy_units))
    ) # 1 / (T * J/T) = 1 / J (inverse self.energy_units).
    print(beta)

Energy units are not compatible with kJ/mol.
2.4143235053466402e+23


In [27]:
print(kT_omm)
print(1/beta)
print(kT_omm == (1/beta))

4.141947e-21 J
4.141947e-21 J
True
